# Charité Dataset - Complete EDA
Exploratory Data Analysis of Charité-Universitätsmedizin Berlin FoG Dataset

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
base_path = Path(r'c:\Users\david\Documents\UADY_CARRERA\10_Decimo_Semestre\Seminario_II\Seminario_II\Datasets\Charité–Universitätsmedizin Berlin\Data Sheet 2\data')

# Column names based on dataset structure
column_names = ['time_s', 'acc_x_m_s2', 'acc_y_m_s2', 'acc_z_m_s2', 
                'gyr_x_deg_s', 'gyr_y_deg_s', 'gyr_z_deg_s', 'fog_label']

# Load all subject files
subject_folders = sorted([f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')])
print(f"Found {len(subject_folders)} subjects")

In [ ]:
dfs = []
for subject_folder in subject_folders:
    subject_id = int(subject_folder.name.replace('S', ''))
    csv_files = sorted(subject_folder.glob('*.csv'))
    
    for csv_file in csv_files:
        # Parse filename: S1_left_foot_trial_1.csv
        parts = csv_file.stem.split('_')
        foot = parts[1]
        trial_id = int(parts[4])
        
        df = pd.read_csv(csv_file)
        df.columns = column_names
        df['subject_id'] = subject_id
        df['trial_id'] = trial_id
        df['foot'] = foot
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(data):,} samples")

## Data Preprocessing

In [ ]:
# Standardize label column
data['label'] = data['fog_label'].astype(int)

# Keep only walking data (all data in Charité is walking-related)
# The dataset contains only FoG during walking activities

# Define sensor features
feature_cols = ['acc_x_m_s2', 'acc_y_m_s2', 'acc_z_m_s2', 
                'gyr_x_deg_s', 'gyr_y_deg_s', 'gyr_z_deg_s']

print(f"Total samples: {len(data):,}")
print(f"Label distribution:")
print(data['label'].value_counts())

In [ ]:
data.head()

In [ ]:
data.info()

## 1. General Dataset Information

In [ ]:
total_samples = len(data)
n_subjects = data['subject_id'].nunique()
n_trials = data.groupby('subject_id')['trial_id'].nunique()

print(f"Total samples: {total_samples:,}")
print(f"Number of subjects: {n_subjects}")
print(f"Trials per subject:")
print(n_trials)
print(f"\nAverage trials per subject: {n_trials.mean():.2f}")
print(f"Average samples per subject: {data.groupby('subject_id').size().mean():.0f}")

## 2. Class Balance

In [ ]:
class_counts = data['label'].value_counts()
class_pct = data['label'].value_counts(normalize=True) * 100

print("Class distribution:")
print(f"No FoG (0): {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"FoG (1): {class_counts[1]:,} ({class_pct[1]:.2f}%)")
print(f"\nImbalance ratio: {class_counts[0] / class_counts[1]:.2f}:1")

In [ ]:
# Distribution by subject
subject_fog_stats = data.groupby('subject_id')['label'].agg(['sum', 'count', 'mean'])
subject_fog_stats.columns = ['fog_samples', 'total_samples', 'fog_ratio']
subject_fog_stats = subject_fog_stats.sort_values('fog_ratio', ascending=False)

print("FoG distribution by subject:")
print(subject_fog_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall class distribution
axes[0].bar(['No FoG', 'FoG'], class_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Number of samples')
axes[0].set_title('Overall Class Distribution')
axes[0].grid(axis='y', alpha=0.3)

# FoG ratio by subject
axes[1].bar(subject_fog_stats.index.astype(str), subject_fog_stats['fog_ratio'], color='#3498db')
axes[1].set_xlabel('Subject')
axes[1].set_ylabel('FoG ratio')
axes[1].set_title('FoG Ratio by Subject')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Temporal Characteristics

In [ ]:
# Sampling frequency
sampling_freq = 200  # Hz (Charité dataset)

# Duration per trial
duration_per_trial = data.groupby(['subject_id', 'trial_id', 'foot'])['time_s'].max()

print(f"Sampling frequency: {sampling_freq} Hz")
print(f"Total recording duration: {duration_per_trial.sum()/3600:.2f} hours")
print(f"Average duration per trial: {duration_per_trial.mean():.2f} seconds")
print(f"Min duration: {duration_per_trial.min():.2f} seconds")
print(f"Max duration: {duration_per_trial.max():.2f} seconds")

In [ ]:
# FoG episode analysis
def get_fog_episodes(df):
    episodes = []
    in_episode = False
    start_idx = None
    
    for idx, label in enumerate(df['label'].values):
        if label == 1 and not in_episode:
            in_episode = True
            start_idx = idx
        elif label == 0 and in_episode:
            in_episode = False
            episodes.append((start_idx, idx - 1))
    
    if in_episode:
        episodes.append((start_idx, len(df) - 1))
    
    return episodes

episode_durations = []
episode_intervals = []
episodes_per_trial = []

for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            trial_data = data[(data['subject_id'] == subject_id) & 
                              (data['trial_id'] == trial_id) & 
                              (data['foot'] == foot)].reset_index(drop=True)
            episodes = get_fog_episodes(trial_data)
            episodes_per_trial.append(len(episodes))
            
            for i, (start, end) in enumerate(episodes):
                duration = (end - start + 1) / sampling_freq
                episode_durations.append(duration)
                
                if i > 0:
                    prev_end = episodes[i-1][1]
                    interval = (start - prev_end) / sampling_freq
                    episode_intervals.append(interval)

if episode_durations:
    print(f"Total FoG episodes: {len(episode_durations)}")
    print(f"Average episode duration: {np.mean(episode_durations):.2f} seconds")
    print(f"Median episode duration: {np.median(episode_durations):.2f} seconds")
    print(f"Std episode duration: {np.std(episode_durations):.2f} seconds")
    print(f"Min episode duration: {np.min(episode_durations):.2f} seconds")
    print(f"Max episode duration: {np.max(episode_durations):.2f} seconds")

if episode_intervals:
    print(f"\nAverage interval between episodes: {np.mean(episode_intervals):.2f} seconds")
    print(f"Median interval: {np.median(episode_intervals):.2f} seconds")

print(f"\nAverage episodes per trial: {np.mean(episodes_per_trial):.2f}")
print(f"Max episodes in a trial: {np.max(episodes_per_trial)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if episode_durations:
    axes[0].hist(episode_durations, bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
    axes[0].axvline(np.mean(episode_durations), color='blue', linestyle='--', label='Mean')
    axes[0].axvline(np.median(episode_durations), color='green', linestyle='--', label='Median')
    axes[0].set_xlabel('Duration (seconds)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('FoG Episode Duration Distribution')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

if episode_intervals:
    axes[1].hist(episode_intervals, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(episode_intervals), color='blue', linestyle='--', label='Mean')
    axes[1].axvline(np.median(episode_intervals), color='green', linestyle='--', label='Median')
    axes[1].set_xlabel('Interval (seconds)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Inter-Episode Interval Distribution')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sensor Analysis

In [ ]:
print("Overall sensor statistics:")
print(data[feature_cols].describe())

In [ ]:
print("Statistics by class:")
for label in [0, 1]:
    print(f"\n{'No FoG' if label == 0 else 'FoG'}:")
    print(data[data['label'] == label][feature_cols].describe())

In [ ]:
# Sample data for visualization (every 500th sample for efficiency)
sample_data = data[feature_cols + ['label']].iloc[::500]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, col in enumerate(feature_cols):
    row = idx // 3
    col_idx = idx % 3
    
    # Distribution
    axes[row, col_idx].hist(sample_data[sample_data['label']==0][col], bins=50, alpha=0.5, label='No FoG', color='#2ecc71')
    axes[row, col_idx].hist(sample_data[sample_data['label']==1][col], bins=50, alpha=0.5, label='FoG', color='#e74c3c')
    axes[row, col_idx].set_xlabel(col)
    axes[row, col_idx].set_ylabel('Frequency')
    axes[row, col_idx].set_title(f'{col} Distribution')
    axes[row, col_idx].legend()
    axes[row, col_idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, col in enumerate(feature_cols):
    row = idx // 3
    col_idx = idx % 3
    
    data_to_plot = [sample_data[sample_data['label']==0][col], 
                    sample_data[sample_data['label']==1][col]]
    axes[row, col_idx].boxplot(data_to_plot, labels=['No FoG', 'FoG'])
    axes[row, col_idx].set_ylabel(col)
    axes[row, col_idx].set_title(f'{col} by Class')
    axes[row, col_idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. FoG Episode Analysis

In [ ]:
# Transition analysis
transitions = (data['label'].diff() != 0).sum()
fog_starts = ((data['label'] == 1) & (data['label'].shift(1) == 0)).sum()
fog_ends = ((data['label'] == 0) & (data['label'].shift(1) == 1)).sum()

print(f"Total transitions: {transitions}")
print(f"FoG episode starts: {fog_starts}")
print(f"FoG episode ends: {fog_ends}")

In [ ]:
# Pre-FoG context (5 seconds before FoG)
pre_fog_window = int(5 * sampling_freq)
pre_fog_data = []

for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            trial_data = data[(data['subject_id'] == subject_id) & 
                              (data['trial_id'] == trial_id) & 
                              (data['foot'] == foot)].reset_index(drop=True)
            fog_starts_idx = trial_data[trial_data['label'].diff() == 1].index
            
            for start_idx in fog_starts_idx:
                if start_idx >= pre_fog_window:
                    pre_fog_segment = trial_data.iloc[start_idx-pre_fog_window:start_idx][feature_cols]
                    pre_fog_data.append(pre_fog_segment.mean())

if pre_fog_data:
    pre_fog_df = pd.DataFrame(pre_fog_data)
    print("Pre-FoG statistics (5s before episode):")
    print(pre_fog_df.describe())

In [ ]:
# Episode duration boxplot
if episode_durations:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.boxplot(episode_durations, vert=False)
    ax.set_xlabel('Duration (seconds)')
    ax.set_title('FoG Episode Duration Distribution')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Sample data for correlation (every 500th sample)
sample_data = data[feature_cols + ['label']].iloc[::500]

correlation_matrix = sample_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix (Sensors + Label)')
plt.tight_layout()
plt.show()

## 7. Variability Analysis

In [ ]:
# Inter-subject variability
subject_stats = data.groupby('subject_id')[feature_cols].agg(['mean', 'std'])

print("Inter-subject variability (coefficient of variation):")
for col in feature_cols:
    cv = subject_stats[(col, 'std')].mean() / abs(subject_stats[(col, 'mean')].mean())
    print(f"{col}: {cv:.3f}")

In [ ]:
# Intra-subject variability
intra_subject_cv = []

for subject in data['subject_id'].unique():
    subject_data = data[data['subject_id'] == subject]
    trial_stats = subject_data.groupby(['trial_id', 'foot'])[feature_cols].mean()
    
    if len(trial_stats) > 1:
        for col in feature_cols:
            cv = trial_stats[col].std() / abs(trial_stats[col].mean())
            intra_subject_cv.append({'Subject': subject, 'Sensor': col, 'CV': cv})

intra_cv_df = pd.DataFrame(intra_subject_cv)
print("Intra-subject variability (average CV across subjects):")
print(intra_cv_df.groupby('Sensor')['CV'].mean())

In [ ]:
# FoG pattern consistency
fog_consistency = data[data['label']==1].groupby('subject_id')[feature_cols].std().mean(axis=1)
print("FoG pattern consistency (lower std = more consistent):")
print(f"Mean std across subjects: {fog_consistency.mean():.3f}")
print(f"Range: {fog_consistency.min():.3f} - {fog_consistency.max():.3f}")

## 8. Data Quality

In [ ]:
# Missing values
missing = data[feature_cols + ['label']].isnull().sum()
print("Missing values:")
print(missing)
print(f"\nTotal missing: {missing.sum()}")

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    return ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

print("Outliers detected (3*IQR method):")
for col in feature_cols:
    n_outliers = detect_outliers_iqr(data, col)
    pct = (n_outliers / len(data)) * 100
    print(f"{col}: {n_outliers:,} ({pct:.2f}%)")

In [ ]:
# Check for discontinuities (large time jumps)
discontinuities = []

for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            trial_data = data[(data['subject_id'] == subject_id) & 
                              (data['trial_id'] == trial_id) & 
                              (data['foot'] == foot)].sort_values('time_s')
            time_diff = trial_data['time_s'].diff()
            jumps = time_diff[time_diff > 0.1].values  # More than 0.1s gap
            discontinuities.extend(jumps)

print(f"Discontinuities (time jumps > 0.1s): {len(discontinuities)}")
if discontinuities:
    print(f"Average jump size: {np.mean(discontinuities):.2f} seconds")
    print(f"Max jump size: {np.max(discontinuities):.2f} seconds")

In [ ]:
# Value range analysis
print("Sensor value ranges:")
for col in feature_cols:
    print(f"{col}: [{data[col].min():.3f}, {data[col].max():.3f}]")

## 9. Key Visualizations

In [ ]:
# Time series with FoG episodes marked
sample_subject = data['subject_id'].iloc[0]
sample_trial = data[data['subject_id'] == sample_subject]['trial_id'].iloc[0]
sample_foot = data[(data['subject_id'] == sample_subject) & 
                   (data['trial_id'] == sample_trial)]['foot'].iloc[0]
sample_data = data[(data['subject_id'] == sample_subject) & 
                   (data['trial_id'] == sample_trial) & 
                   (data['foot'] == sample_foot)].iloc[:5000].reset_index(drop=True)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for idx in range(3):
    col = feature_cols[idx]
    axes[idx].plot(sample_data['time_s'], sample_data[col], linewidth=0.5, color='#34495e')
    
    fog_periods = sample_data[sample_data['label'] == 1]
    for _, row in fog_periods.iterrows():
        axes[idx].axvline(row['time_s'], color='red', alpha=0.3, linewidth=0.8)
    
    axes[idx].set_ylabel(col)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_title(f'{col} Signal with FoG Episodes')

# Label visualization
axes[3].plot(sample_data['time_s'], sample_data['label'], linewidth=1, color='#e74c3c')
axes[3].set_ylabel('Label')
axes[3].set_xlabel('Time (seconds)')
axes[3].set_title('FoG Label')
axes[3].grid(alpha=0.3)
axes[3].set_yticks([0, 1])
axes[3].set_yticklabels(['No FoG', 'FoG'])

plt.tight_layout()
plt.show()

In [ ]:
# Episode duration boxplot by subject
episode_by_subject = []

for subject_id in data['subject_id'].unique():
    for trial_id in data[data['subject_id'] == subject_id]['trial_id'].unique():
        for foot in data[(data['subject_id'] == subject_id) & 
                         (data['trial_id'] == trial_id)]['foot'].unique():
            trial_data = data[(data['subject_id'] == subject_id) & 
                              (data['trial_id'] == trial_id) & 
                              (data['foot'] == foot)].reset_index(drop=True)
            episodes = get_fog_episodes(trial_data)
            
            for start, end in episodes:
                duration = (end - start + 1) / sampling_freq
                episode_by_subject.append({'Subject': subject_id, 'Duration': duration})

if episode_by_subject:
    episode_df = pd.DataFrame(episode_by_subject)
    
    plt.figure(figsize=(12, 6))
    episode_df.boxplot(column='Duration', by='Subject', figsize=(12, 6))
    plt.ylabel('Duration (seconds)')
    plt.xlabel('Subject')
    plt.title('FoG Episode Duration by Subject')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## Export Dataset to CSV

In [ ]:
output_path = Path(r'c:\Users\david\Documents\UADY_CARRERA\10_Decimo_Semestre\Seminario_II\Seminario_II\outputs\datasets_csv')
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / 'charite_complete_dataset.csv'
data.to_csv(output_file, index=False)

print(f"Dataset exported to: {output_file}")
print(f"Total samples: {len(data):,}")
print(f"File size: {output_file.stat().st_size / (1024**2):.2f} MB")